# LSTM & GRU Models — Experiment Notebook
**Task 3: Deep Learning Time Series Forecasting**

This notebook:
1. Loads preprocessed + scaled data
2. Trains LSTM and GRU on each stock
3. Walk-forward test evaluation
4. Forecasts next 5 trading days
5. Compares LSTM vs GRU performance

In [ ]:
import sys
sys.path.append("..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data.fetch_data import load_all_raw, STOCK_UNIVERSE
from src.data.preprocess import preprocess_all
from src.utils.metrics import evaluate
from src.models.lstm import (
    LSTMModel, GRUModel, train_model,
    predict_test_rnn, forecast_future_rnn,
    run_lstm_pipeline, run_gru_pipeline
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
print(f"PyTorch {torch.__version__} | Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Load Data

In [ ]:
raw = load_all_raw()
processed = preprocess_all(raw, save=False)
print(f"Loaded {len(processed)} stocks")

## 2. Single-Stock LSTM Deep Dive (TCS.NS)

In [ ]:
TICKER = "TCS.NS"
SEQ_LEN = 60
EPOCHS = 30

d = processed[TICKER]
train, test = d["train"], d["test"]
tr_sc, te_sc, scaler = d["train_scaled"], d["test_scaled"], d["scaler"]

print(f"Train scaled shape: {tr_sc.shape}")
print(f"Test scaled shape:  {te_sc.shape}")

In [ ]:
# Train LSTM
lstm_model = LSTMModel(hidden_size=64)
losses = train_model(lstm_model, tr_sc, seq_len=SEQ_LEN, epochs=EPOCHS)

# Training loss curve
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color="steelblue")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title(f"{TICKER} — LSTM Training Loss")
plt.tight_layout()
plt.show()

In [ ]:
# Walk-forward test predictions
raw_preds = predict_test_rnn(lstm_model, tr_sc, te_sc, scaler, SEQ_LEN)
pred = pd.Series(raw_preds, index=test.index, name="LSTM_Pred")

metrics = evaluate(test.values, pred.values, "LSTM", TICKER)
print(f"\nLSTM Metrics for {TICKER}:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train[-100:], label="Train", color="steelblue", alpha=0.5)
ax.plot(test, label="Actual", color="green", linewidth=2)
ax.plot(pred, label="LSTM", color="purple", linewidth=1.5, linestyle="--")
ax.set_title(f"{TICKER} — LSTM Forecast vs Actual")
ax.legend()
plt.tight_layout()
plt.show()

### Future Forecast

In [ ]:
fc = forecast_future_rnn(lstm_model, tr_sc, te_sc, scaler, SEQ_LEN, n_periods=5)
full = pd.concat([train, test])
future_dates = pd.bdate_range(start=full.index[-1] + pd.Timedelta(days=1), periods=5)

print(f"\nLSTM Forecast — {TICKER}:")
for d_val, v in zip(future_dates, fc):
    print(f"  {d_val.date()}: ₹{v:.2f}")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(full[-60:], label="Historical", color="steelblue")
ax.plot(future_dates, fc, label="LSTM Forecast", color="purple", marker="o")
ax.set_title(f"{TICKER} — LSTM 5-Day Ahead Forecast")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Run LSTM on All Stocks

In [ ]:
lstm_preds, lstm_fc, lstm_met = run_lstm_pipeline(processed, n_forecast=5, epochs=EPOCHS)

lstm_df = pd.DataFrame(lstm_met)
print("\n── LSTM Results ──")
print(lstm_df.to_string())
print(f"\nAvg MAPE: {lstm_df['MAPE'].mean():.2f}%  |  Avg DirAcc: {lstm_df['DirAcc'].mean():.1f}%")

## 4. Run GRU on All Stocks

In [ ]:
gru_preds, gru_fc, gru_met = run_gru_pipeline(processed, n_forecast=5, epochs=EPOCHS)

gru_df = pd.DataFrame(gru_met)
print("\n── GRU Results ──")
print(gru_df.to_string())
print(f"\nAvg MAPE: {gru_df['MAPE'].mean():.2f}%  |  Avg DirAcc: {gru_df['DirAcc'].mean():.1f}%")

## 5. LSTM vs GRU Comparison

In [ ]:
combined = pd.concat([lstm_df, gru_df])
comparison = combined.groupby("Model")[["RMSE", "MAPE", "DirAcc"]].mean().round(4)
print("\n── LSTM vs GRU (Avg Across Stocks) ──")
print(comparison.to_string())

# Side-by-side bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, metric in enumerate(["RMSE", "MAPE", "DirAcc"]):
    comparison[metric].plot(kind="bar", ax=axes[i], color=["purple", "teal"])
    axes[i].set_title(metric)
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", rotation=0)
plt.suptitle("LSTM vs GRU Comparison", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# All-stock prediction grids
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()
for i, ticker in enumerate(list(lstm_preds.keys())[:9]):
    ax = axes[i]
    actual = processed[ticker]["test"]
    ax.plot(actual, label="Actual", color="green", linewidth=1.5)
    ax.plot(lstm_preds[ticker], label="LSTM", color="purple", linewidth=1, linestyle="--")
    if ticker in gru_preds:
        ax.plot(gru_preds[ticker], label="GRU", color="teal", linewidth=1, linestyle=":")
    ax.set_title(STOCK_UNIVERSE.get(ticker, ticker), fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=7)
    ax.legend(fontsize=7)
plt.suptitle("LSTM vs GRU — All Stocks", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()